### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build knowledge graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
vocab = feature_engineer.vocab2idx
train_hetero_graph = experiment_data_preprocessor.create_knowledge_graph(encoded_train_df, vocab)


Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building user-item edges...
Building item-attribute edges...
Knowledge Graph: HeteroData(
  user={ num_nodes=2065 },
  movie={ num_nodes=8707 },
  actor={ num_nodes=2287 },
  country={ num_nodes=42 },
  director={ num_nodes=542 },
  genre={ num_nodes=22 },
  (user, interacts_with, movie)={ edge_index=[2, 158984] },
  (movie, interacts_with, user)={ edge_index=[2, 158984] },
  (movie, has_actor, actor)={ edge_index=[2, 33520] },
  (actor, has_actor, movie)={ edge_index=[2, 33520] },
  (movie, has_country, country)={ edge_index=[2, 6704] },
  (country, has_country, movie)={ edge_index=[2, 6704] },
  (movie, has_director, director)={ edge_index=[2, 6704] },
  (director, has_director, movie)={ edge_index=[2, 6704] },
  (movie, has_genre, genre)={ edge_index=[2, 53632] },
  (genre, has_genre, movie)={ edge_index=[2, 53632] }
)
Node Type: ['user', 'movie', 'actor', '

#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:05<00:00, 347.80it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, ...",0.577311,"[0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, ...",0.183214,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.494533,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


#### Prepare Item Multihot Vec on Each Dimension

In [9]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1588,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
# train_triplet_with_dps_df.head(1)

valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
valid_triplet_with_dps_df = valid_triplet_df.merge(user_dps_df, on="userID", how="inner")
valid_triplet_with_dps_df = valid_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="inner")
valid_triplet_with_dps_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920
Original data count (positive samples): 18261
Num of triplets: 18261(pos samples) * 5(negative sampled items) = 91305


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,962,4662,"[328, 704, 1, 1, 1]",37,464,"[2, 3, 6, 10, 12, 0, 0, 0]","[988, 648, 2087, 1, 924]",37,75,...,0.183214,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.494533,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545,962,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ..."


#### Prepare prediction pool for inference/testing

In [11]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[2136, 281, 1446, 61, 1]",36,1,"[9, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[1578, 466, 911, 941, 887]",37,432,"[2, 19, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[1, 1, 1, 1, 1]",37,1,"[12, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[1, 1383, 554, 1, 828]",37,534,"[9, 16, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[446, 1010, 637, 809, 547]",36,70,"[7, 12, 15, 18, 0, 0, 0, 0]"


In [12]:
train_triplet_with_dps_df.head(1)

,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,1102,307,"[2267, 1401, 582, 998, 1847]",37,360,"[2, 3, 17, 18, 0, 0, 0, 0]","[1769, 713, 931, 1356, 1]",12,452,...,0.183214,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.494533,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545,1102,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


### Prepare DataLoader

In [13]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping, validate_unique_pairs

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
valid_dataset = TripletDataset(valid_triplet_with_dps_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

# TODO: Custom sampler
MIN_POS_ITEMS = 2
user_to_pos_items, user_pos_to_indices = get_user_triplet_mapping(train_triplet_with_dps_df, MIN_POS_ITEMS)
train_sampler = UserPosItemSampler(user_to_pos_items, user_pos_to_indices, batch_size=BATCH_SIZE, min_pos_items=MIN_POS_ITEMS, max_pos_items=20, buffer=0)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, worker_init_fn=seed_worker, generator=g)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 90805
test data count: 1032000


In [14]:
# from common.datasets import validate_unique_pairs
# print(len(train_loader))
# for i in train_loader:
#     user = i["user"]
#     pos_item = i["pos_item"]
#     validate_unique_pairs(user, pos_item)
#     break

### Configure Model (LightningModule)

In [15]:
# from lightning_models.exp.mtdp_kgat_v2_loss_weight_test import MTDPRec
from lightning_models.extensions.mtdp_kgat_v2 import MTDPRec

EMB_DIM = 64
REL_EMB_DIM = 16
LR = 1e-3
EPOCHS = 5
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

NUM_NEIGHBORS = 10
USE_MINI_BATCH = False
NODE_DROPOUT_RATE = 0
MESS_DROPOUT_RATE = 0.2

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

# TODO: determine the static weights for multi-task learning
MT_WEIGHTS = {
    "bpr_loss": 1.0,
    "kg_loss": 1.0,
    "dps_loss": 0.5,
    "dpr_loss": 0.5,
    "dpm_loss": 0.5,
}

RESCALE_METHOD = "ema"  # None, "log", "ema"

model = MTDPRec(
    hetero_data=train_hetero_graph,
    embedding_dim=EMB_DIM,
    rel_emb_dim=REL_EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=NODE_DROPOUT_RATE,
    mess_dropout=MESS_DROPOUT_RATE,
    num_neighbors=NUM_NEIGHBORS,
    lr=LR,
    reg_weight=REG_WEIGHT,
    use_mini_batch=USE_MINI_BATCH,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD, # TODO: loss rescale method (None, "log", "ema")
)


Seed set to 42


### Configure Trainer and Experiment

In [16]:
from common._mlflow import get_mlflow_logger, get_callbacks

# TODO: set experiment name, version, run name, and patience
EXPERIMENT_NAME = "mtdp-kgat-v2-exp"
VERSION = "custom"
RUN_NAME = "ema"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_loss",
    monitor_mode="min",
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}",
)

In [17]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [18]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/mtdp-kgat-v2-exp exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                | Type              | Params | Mode 
-------------------------------------------------------------------
0  | kgat_model          | KGAT              | 888 K  | train
1  | bpr_loss            | BPRLoss           | 0      | train
2  | reg_loss            | EmbLoss           | 0      | train
3  | dps_module          | DPSPredictor      | 1.0 K  | train
4  | dps_loss_fn         | DPSLoss           | 0      | train
5  | dpr_module          | DPRegularizer     | 263 K  | train
6  | dpr_loss_fn         | DPRLoss           | 0      | train
7  | dpm_module          | DPMatcher         | 0      | train
8  | dpm_loss_fn         | KLDivergenceLoss  | 0      | train
9  | loss_scaling_module | LogS

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss = inf is not finite. Previous best value was inf. Signaling Trainer to stop.
Epoch 0, global step 775: 'val_loss' reached inf (best inf), saving model to '/home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/mtdp-kgat-v2-exp/[custom]-ema-emb_dim=64-num_layers=3-lr=0.001-best-checkpoint-epoch=00-val_loss=inf.ckpt' as top 1


🏃 View run ema at: http://140.112.106.216:3683/#/experiments/13/runs/4eeaa4b5f080460dbf500769b49cfb17
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/13


### Inference

In [19]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "mtdp-kgat-v2-loss-weight-exp"
# best_model_checkpoint_path = "[log-scale]-run02-emb_dim=64-num_layers=3-lr=0.001-best-checkpoint-epoch=02-val_loss=3.83.ckpt"
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = MTDPRec.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.3453719913959503     │
│        test_ndcg20        │    0.3811015784740448     │
│        test_ndcg5         │    0.2917705476284027     │
│     test_precision10      │    0.12088178098201752    │
│     test_precision20      │    0.11128875613212585    │
│      test_precision5      │    0.12810078263282776    │
│       test_recall10       │    0.10339446365833282    │
│       test_recall20       │    0.1811717003583908     │
│       test_recall5        │    0.05677869915962219    │
└───────────────────────────┴───────────────────────────┘

🏃 View run ema at: http://140.112.106.216:3683/#/experiments/13/runs/4eeaa4b5f080460dbf500769b49cfb17
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/13


[{'test_ndcg5': 0.2917705476284027,
  'test_ndcg10': 0.3453719913959503,
  'test_ndcg20': 0.3811015784740448,
  'test_precision5': 0.12810078263282776,
  'test_precision10': 0.12088178098201752,
  'test_precision20': 0.11128875613212585,
  'test_recall5': 0.05677869915962219,
  'test_recall10': 0.10339446365833282,
  'test_recall20': 0.1811717003583908}]

In [20]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.291771,0.056779,0.128101,0.345372,0.103394,0.120882,0.381102,0.181172,0.111289
std,595.969798,0.359439,0.099422,0.171292,0.320973,0.137043,0.133614,0.274567,0.177907,0.106507
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.231378,0.040000,0.050000
50%,1031.500000,0.000000,0.000000,0.000000,0.356207,0.058824,0.100000,0.386853,0.142857,0.100000
75%,1547.250000,0.570642,0.081644,0.200000,0.572399,0.153846,0.200000,0.561497,0.260870,0.150000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.700000,1.000000,1.000000,0.650000


In [21]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=20,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

exploded 2064
extracting item features...
merging features...
interaction data count before merging: 41280
interaction data count after merging: 41280
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:03<00:00, 599.29it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.168442,0.971310,0.167602,0.879606,0.546740
std,595.969798,0.074718,0.036428,0.101726,0.074162,0.052887
min,0.000000,0.000000,0.369298,0.000000,0.516043,0.287811
25%,515.750000,0.113382,0.964899,0.090994,0.848866,0.511423
50%,1031.500000,0.171370,0.982318,0.162891,0.898494,0.550756
75%,1547.250000,0.227408,0.991207,0.235396,0.931832,0.585018
max,2063.000000,0.389181,0.999447,0.509021,0.986819,0.691028
